In [1]:
from simphony.libraries.ideal.s_parameters import optical_s_parameter
from simphony.libraries.siepic import y_branch
import sax
import jax.numpy as jnp

from simphony.libraries.ideal.filters import discrete_state_space

# state_space_model = discrete_state_space(2, 2)
def coupler():
    return {
        ("in0@TE", "out0@TE"): 0.45**0.5,
        ("in0@TE", "out1@TE"): 1j * 0.45**0.5,
        ("in1@TE", "out0@TE"): 1j * 0.45**0.5,
        ("in1@TE", "out1@TE"): 0.45**0.5,
        ("in0@TM", "out0@TM"): 0.45**0.5,
        ("in0@TM", "out1@TM"): 1j * 0.45**0.5,
        # ("in1@TM", "out0@TM"): 1j * 0.45**0.5,
        # ("in1@TM", "out1@TM"): 0.45**0.5,
        ("in0@TE", "out0@TM"): 0.01**0.5,
        ("in0@TE", "out1@TM"): 1j * 0.01**0.5,
        ("in1@TE", "out0@TM"): 1j * 0.01**0.5,
        ("in1@TE", "out1@TM"): 0.01**0.5,
        ("in0@TM", "out0@TE"): 0.01**0.5,
        ("in0@TM", "out1@TE"): 1j * 0.01**0.5,
        # ("in1@TM", "out0@TE"): 1j * 0.01**0.5,
        # ("in1@TM", "out1@TE"): 0.01**0.5,
    }

N = 4

A = jnp.eye(N, k=-1)
B = jnp.zeros((N, 1)).at[0, 0].set(1.0)

C = jnp.array([[0.5, 0.3, 0.1, 0.05]])
D = jnp.array([[0.2]])

# state_space_model(A, B, C, D)

pcell = optical_s_parameter(y_branch, port_directionality={'port_1': 'output'})
# pcell = optical_s_parameter(coupler)
pcell()

<simphony.libraries.ideal.s_parameters.optical_s_parameter.<locals>.SParameterSax at 0x73a8b3230470>

In [3]:
def outer(a, b={}):
    class Inner:
        def __init__(self, c=3):
            print(a, b, c)
    return Inner

X = outer(1)
obj = X()  # prints: 1 2 3


1 {} 3


In [2]:
from simphony.libraries.ideal.s_parameters import optical_s_parameter
import sax
import jax.numpy as jnp

def waveguide(wl=1.55, wl0=1.55, neff=2.34, ng=3.4, length=10.0, loss=0.0):
    """A simple straight waveguide model

    Args:
        wl: wavelength
        neff: waveguide effective index
        ng: waveguide group index (used for linear neff dispersion)
        wl0: center wavelength at which neff is defined
        length: [m] wavelength length
        loss: [dB/m] waveguide loss
    """
    dwl = wl - wl0
    dneff_dwl = (ng - neff) / wl0
    neff = neff - dwl * dneff_dwl
    phase = 2 * jnp.pi * neff * length / wl
    transmission = 10 ** (-loss * length / 20) * jnp.exp(1j * phase)
    sdict = sax.reciprocal(
        {
            ("in0@TE", "out0@TE"): 0.95 * transmission,  # 5% lost to cross-polarization
            ("in0@TE", "out0@TM"): 0.05 * transmission,  # 5% cross-polarization
            ("in0@TM", "out0@TM"): 0.85 * transmission,  # 10% worse tm->tm than te->te
            ("in0@TM", "out0@TE"): 0.05 * transmission,  # 5% cross-polarization
        }
    )
    return sdict

def coupler():
    return {
        ("in0@TE", "out0@TE"): 0.45**0.5,
        ("in0@TE", "out1@TE"): 1j * 0.45**0.5,
        ("in1@TE", "out0@TE"): 1j * 0.45**0.5,
        ("in1@TE", "out1@TE"): 0.45**0.5,
        ("in0@TM", "out0@TM"): 0.45**0.5,
        ("in0@TM", "out1@TM"): 1j * 0.45**0.5,
        # ("in1@TM", "out0@TM"): 1j * 0.45**0.5,
        # ("in1@TM", "out1@TM"): 0.45**0.5,
        ("in0@TE", "out0@TM"): 0.01**0.5,
        ("in0@TE", "out1@TM"): 1j * 0.01**0.5,
        ("in1@TE", "out0@TM"): 1j * 0.01**0.5,
        ("in1@TE", "out1@TM"): 0.01**0.5,
        ("in0@TM", "out0@TE"): 0.01**0.5,
        ("in0@TM", "out1@TE"): 1j * 0.01**0.5,
        # ("in1@TM", "out0@TE"): 1j * 0.01**0.5,
        # ("in1@TM", "out1@TE"): 0.01**0.5,
    }

mzi, _ = sax.circuit(
    netlist={
        "instances": {
            "lft": "coupler",  # single mode models will be automatically converted to multimode models without cross polarization.
            "top": {"component": "straight", "settings": {"length": 25.0}},
            "btm": {"component": "straight", "settings": {"length": 15.0}},
            "rgt": "coupler",  # single mode models will be automatically converted to multimode models without cross polarization.
        },
        "connections": {
            "lft,out0": "btm,in0",
            "btm,out0": "rgt,in0",
            "lft,out1": "top,in0",
            "top,out0": "rgt,in1",
        },
        "ports": {
            "in0": "lft,in0",
            "in1": "lft,in1",
            "out0": "rgt,out0",
            "out1": "rgt,out1",
        },
    },
    models={
        "coupler": coupler,
        "straight": waveguide,
    },
)

mzi()

{('in0@TE', 'in0@TE'): Array(0.+0.j, dtype=complex128),
 ('in0@TE', 'in0@TM'): Array(0.+0.j, dtype=complex128),
 ('in0@TE', 'in1@TE'): Array(0.+0.j, dtype=complex128),
 ('in0@TE', 'out0@TE'): Array(-0.24916194+0.0802182j, dtype=complex128),
 ('in0@TE', 'out0@TM'): Array(-0.08473572-0.04952667j, dtype=complex128),
 ('in0@TE', 'out1@TE'): Array(0.78038401-0.29280674j, dtype=complex128),
 ('in0@TE', 'out1@TM'): Array(0.17781767-0.0912419j, dtype=complex128),
 ('in0@TM', 'in0@TE'): Array(0.+0.j, dtype=complex128),
 ('in0@TM', 'in0@TM'): Array(0.+0.j, dtype=complex128),
 ('in0@TM', 'in1@TE'): Array(0.+0.j, dtype=complex128),
 ('in0@TM', 'out0@TE'): Array(-0.08362144-0.02755491j, dtype=complex128),
 ('in0@TM', 'out0@TM'): Array(-0.24340063-0.30245117j, dtype=complex128),
 ('in0@TM', 'out1@TE'): Array(0.19978943-0.09235618j, dtype=complex128),
 ('in0@TM', 'out1@TM'): Array(0.32812638-0.24470273j, dtype=complex128),
 ('in1@TE', 'in0@TE'): Array(0.+0.j, dtype=complex128),
 ('in1@TE', 'in0@TM'):

In [5]:



port_directionality = {
    'port_1': 'input', 
    # 'port_2': 'input', 
    'port_3': 'output'
}

optical_s_parameter(y_branch, port_directionality=port_directionality)

simphony.libraries.ideal.s_parameters.optical_s_parameter.<locals>.SParameterSax

In [10]:
sax.multimode(mzi)() == sax.multimode(mzi, modes=("TESTMODE", "TESTMODE2"))() 

True